# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the short exercises in [06-geospatial-vector-data-exercises.ipynb](06-geospatial-vector-data-exercises.ipynb). Exercise 8, the real-dataset walkthrough, has no solution provided.
:::

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
gdf = gpd.GeoDataFrame({"name": ["A", "B"]},
                       geometry=[Point(7.4, 46.9), Point(8.5, 47.4)],
                       crs="EPSG:4326")
print(gdf.crs.to_epsg())
print(gdf.geometry.tolist())

## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [ ]:
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
gdf.to_file("_files/points.geojson", driver="GeoJSON")
back = gpd.read_file("_files/points.geojson")
print(back.shape, back.crs.to_epsg())

## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [ ]:
proj = gdf.to_crs(2056)
print(proj.crs.to_epsg())
print(round(proj.geometry.x.iloc[0]), round(proj.geometry.y.iloc[0]))

## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [ ]:
proj = gdf.to_crs(2056)                                  # metres, not degrees
d_m = proj.geometry.iloc[0].distance(proj.geometry.iloc[1])
print(round(d_m / 1000, 2), "km")
# in EPSG:4326 .distance() subtracts angles, giving degrees, which are not a length

## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [ ]:
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
hit = gpd.sjoin(gdf, poly, predicate="within", how="inner")
print(hit["name"].tolist())     # only A lies inside

## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
proj = gdf.to_crs(2056)
buf = proj.buffer(10_000)
print(round(buf.area.iloc[0] / 1e6, 1), "km^2")   # ~314

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
from shapely.geometry import Polygon
zones = gpd.GeoDataFrame({"type": ["flood", "flood"]},
    geometry=[Polygon([(7, 47), (8, 47), (8, 48), (7, 48)]),
              Polygon([(8, 47), (9, 47), (9, 48), (8, 48)])], crs="EPSG:4326")
merged = zones.dissolve(by="type")
print(len(merged))     # 1